# Module 03 — ML for Forecasting

## Notebook 2 · Feature Engineering for ML Forecasting

**Goals**

1. Understand the difference between **causal** and **non-causal** features and why it matters.
2. Build **lag**, **rolling**, and **EWM** features without leakage, *per forecast key*.
3. Add **calendar** features (and optional cyclical encodings).
4. Encode categorical forecast keys efficiently for LightGBM.
5. Wrap everything in a single `FeatureEngineer` instance for reproducibility.

**Why this matters.** Tree-based models like LightGBM cannot extrapolate — they need features that surface the temporal structure for them. A well-engineered feature set is far more impactful than swapping algorithms.


In [1]:
# === Colab / local setup ====================================================
# 1. Install dependencies (uncomment the pip line on first Colab run).
# !pip install -q lightgbm==4.* prophet plotly optuna shap pandas numpy scikit-learn pyarrow

# 2. Make the `utils` package importable. Two options:
#    (a) Notebook is sitting next to a `utils/` folder (recommended).
#    (b) The package is uploaded as a zip; unzip it and `sys.path.append(...)`.
import os, sys
HERE = os.path.dirname(os.path.abspath("__file__"))  # may be empty in Colab
for cand in [".", "..", "/content", "/content/ml_forecasting_tutorial"]:
    if os.path.isdir(os.path.join(cand, "utils")):
        sys.path.insert(0, cand)
        break

# 3. Standard imports for every notebook.
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "colab"   # works in Colab + Jupyter

# 4. Tell pandas to display nicely.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


In [2]:
# === Dataset configuration ==================================================
# EDIT THIS CELL TO POINT AT YOUR DATASET
DATA_PATH    = "./dataset/m5/m5_small.csv"             # path to your CSV / parquet
DATE_COL     = "date"                              # date column
TARGET_COL   = "sales"                          # target / forecast column
KEY_COLS     = ['item_id',
                'dept_id',
                'cat_id',
                'store_id',
                'state_id']                           # columns identifying a unique series
FREQ         = "D"                              # 'D'=daily, 'W'=weekly, 'M'=monthly
HOLDOUT_DAYS = 28                               # length of test horizon


In [3]:
from utils.data_utils import load_forecasting_data, time_based_split, summarize_split

df = load_forecasting_data(DATA_PATH, DATE_COL, TARGET_COL, KEY_COLS)
cutoff = df[DATE_COL].max() - pd.Timedelta(days=HOLDOUT_DAYS - 1)
train_df, test_df = time_based_split(df, DATE_COL, cutoff)
summarize_split(train_df, test_df, DATE_COL)


  train: 8,070,491 rows  |  2015-03-08 → 2016-05-22
  test :  512,232 rows  |  2016-05-23 → 2016-06-19


### 1. Causal vs. non-causal features — the leakage trap

A feature is **causal** at time *t* if it can be computed from information available *strictly before t* (or, for known-in-advance variables, from a deterministic function of the date — e.g., the calendar).

> **The single most common bug** in ML forecasting tutorials is computing a rolling mean *that includes the current day*. This is non-causal: at production time you would not have today's value when predicting today.

The helpers in `utils.feature_engineering` always shift by `min_lag` (default 1) **before** computing rolling statistics. The cell below shows you exactly what that does.


In [4]:
import pandas as pd

# Tiny demo: build a non-causal vs causal 7-day mean for one key
demo = (df.sort_values([*KEY_COLS, DATE_COL]).groupby(KEY_COLS).head(20).copy())
g = demo.groupby(KEY_COLS, sort=False)[TARGET_COL]

demo["roll7_naive_LEAKING"]  = g.transform(lambda s: s.rolling(7, min_periods=1).mean())   # WRONG
demo["roll7_causal"]         = g.transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())  # RIGHT

demo[[DATE_COL, *KEY_COLS, TARGET_COL, "roll7_naive_LEAKING", "roll7_causal"]].head(15)


,date,item_id,dept_id,cat_id,store_id,state_id,sales,roll7_naive_LEAKING,roll7_causal
0,2015-03-08,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,1.0,1.000000,NaN
1,2015-03-09,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,1.0,1.000000,1.000000
2,2015-03-10,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,0.0,0.666667,1.000000
3,2015-03-11,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,0.0,0.500000,0.666667
4,2015-03-12,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,0.0,0.400000,0.500000
5,2015-03-13,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,0.0,0.333333,0.400000
6,2015-03-14,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,1.0,0.428571,0.333333
7,2015-03-15,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,0.0,0.285714,0.428571
8,2015-03-16,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,0.0,0.142857,0.285714
9,2015-03-17,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,0.0,0.142857,0.142857


Notice how the **leaking** version equals the target itself on the first non-NaN row — that's a perfect data leak. The **causal** version is NaN on day 1 and then trails the target by one day.


### 2. Lag features

A lag feature at time *t* is simply the target *k* periods ago: $$y_{t-k}$$. For daily retail data, popular choices are 1, 7, 14, 28 (one day, one week, two weeks, four weeks).

`add_lag_features` is groupby-aware so lags never leak across forecast keys.


In [5]:
from utils.feature_engineering import add_lag_features

train_lagged = add_lag_features(train_df, KEY_COLS, TARGET_COL, lags=[1, 7, 14, 28])
train_lagged.head()


,id,date,item_id,dept_id,cat_id,store_id,state_id,d,sales,sell_price,snap_CA,snap_TX,snap_WI,weight,event_name_1_Chanukah End,event_name_1_Christmas,event_name_1_Cinco De Mayo,event_name_1_ColumbusDay,event_name_1_Easter,event_name_1_Eid al-Fitr,event_name_1_EidAlAdha,event_name_1_Father's day,event_name_1_Halloween,event_name_1_IndependenceDay,event_name_1_LaborDay,...,event_name_1_NewYear,event_name_1_OrthodoxChristmas,event_name_1_OrthodoxEaster,event_name_1_Pesach End,event_name_1_PresidentsDay,event_name_1_Purim End,event_name_1_Ramadan starts,event_name_1_StPatricksDay,event_name_1_SuperBowl,event_name_1_Thanksgiving,event_name_1_ValentinesDay,event_name_1_VeteransDay,event_name_2_Cinco De Mayo,event_name_2_Father's day,event_name_2_OrthodoxEaster,event_type_1_Cultural,event_type_1_National,event_type_1_Religious,event_type_1_Sporting,event_type_2_Cultural,event_type_2_Religious,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28
0,FOODS_1_001_TX_1_evaluation,2015-03-08,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,1500,1.0,2.24,1,0,1,0.000015,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,NaN,NaN,NaN,NaN
1,FOODS_1_001_TX_1_evaluation,2015-03-09,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,1501,1.0,2.24,1,1,1,0.000015,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,1.0,NaN,NaN,NaN
2,FOODS_1_001_TX_1_evaluation,2015-03-10,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,1502,0.0,2.24,1,0,0,0.000015,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,1.0,NaN,NaN,NaN
3,FOODS_1_001_TX_1_evaluation,2015-03-11,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,1503,0.0,2.24,0,1,1,0.000015,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,0.0,NaN,NaN,NaN
4,FOODS_1_001_TX_1_evaluation,2015-03-12,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,1504,0.0,2.24,0,1,1,0.000015,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,0.0,NaN,NaN,NaN


### 3. Rolling statistics

Rolling means / std / min / max over the *lagged* series help the tree split cleanly on level changes that simple lags miss.


In [6]:
from utils.feature_engineering import add_rolling_features

train_lagged_rolled = add_rolling_features(
    train_lagged, KEY_COLS, TARGET_COL,
    windows=[7, 14, 28], stats=["mean", "std", "max"], min_lag=1,
)
[c for c in train_lagged_rolled.columns if "roll" in c]


['sales_roll_mean_7',
 'sales_roll_std_7',
 'sales_roll_max_7',
 'sales_roll_mean_14',
 'sales_roll_std_14',
 'sales_roll_max_14',
 'sales_roll_mean_28',
 'sales_roll_std_28',
 'sales_roll_max_28']

### 4. Exponentially-weighted means

Useful when you believe the *recent* past matters more than the distant past in a smooth fashion. `halflife` is in periods (days here).


In [7]:
from utils.feature_engineering import add_ewm_features

train_with_ewm = add_ewm_features(train_lagged_rolled, KEY_COLS, TARGET_COL, halflives=[7.0, 28.0])
[c for c in train_with_ewm.columns if "ewm" in c]


['sales_ewm_hl7.0', 'sales_ewm_hl28.0']

### 5. Calendar features

Calendar features are **always causal** because they are deterministic functions of the date. We add:

* integer encodings: `year, quarter, month, week, day, day_of_week, day_of_year`,
* boolean flags: `is_weekend, is_month_start, is_month_end, is_quarter_end`,
* optional sin/cos encodings for cyclical features (helpful for linear models; trees handle integers fine).


In [8]:
from utils.feature_engineering import add_calendar_features

train_full = add_calendar_features(train_with_ewm, DATE_COL, cyclical=True)
[c for c in train_full.columns if c not in train_with_ewm.columns]


['year',
 'quarter',
 'month',
 'week',
 'day',
 'day_of_week',
 'day_of_year',
 'is_weekend',
 'is_month_start',
 'is_month_end',
 'is_quarter_end',
 'dow_sin',
 'dow_cos',
 'month_sin',
 'month_cos',
 'doy_sin',
 'doy_cos']

### 6. Categorical encoding

LightGBM has a built-in optimised algorithm for categorical features (Fisher 1958). Setting the dtype to `pandas.Categorical` and passing the column names via `categorical_feature=` is faster *and* more accurate than one-hot encoding for high-cardinality keys like `(store, item)`.


In [9]:
from utils.feature_engineering import encode_categoricals

train_full = encode_categoricals(train_full, KEY_COLS, method="category")
train_full[KEY_COLS].dtypes


item_id     category
dept_id     category
cat_id      category
store_id    category
state_id    category
dtype: object

### 7. The all-in-one `FeatureEngineer`

We will use this object everywhere from now on. It locks in the configuration and applies the same transformations to train and test, so the schemas always match.


In [11]:
from utils.feature_engineering import FeatureEngineer

fe = FeatureEngineer(
    date_col=DATE_COL,
    target_col=TARGET_COL,
    key_cols=KEY_COLS,
    lags=[1, 2, 7, 14, 28],
    rolling_windows=[7, 14, 28],
    rolling_stats=["mean", "std", "max"],
    ewm_halflives=[7.0, 28.0],
    cyclical_calendar=True,
    encode_keys_as_category=True,
)

train_features = fe.transform(train_df)
print(f"train_features shape: {train_features.shape}")
print(f"feature count        : {len(fe.feature_names)}")
fe.feature_names


train_features shape: (8070491, 86)
feature count        : 84


['id',
 'item_id',
 'dept_id',
 'cat_id',
 'store_id',
 'state_id',
 'd',
 'sell_price',
 'snap_CA',
 'snap_TX',
 'snap_WI',
 'weight',
 'event_name_1_Chanukah End',
 'event_name_1_Christmas',
 'event_name_1_Cinco De Mayo',
 'event_name_1_ColumbusDay',
 'event_name_1_Easter',
 'event_name_1_Eid al-Fitr',
 'event_name_1_EidAlAdha',
 "event_name_1_Father's day",
 'event_name_1_Halloween',
 'event_name_1_IndependenceDay',
 'event_name_1_LaborDay',
 'event_name_1_LentStart',
 'event_name_1_LentWeek2',
 'event_name_1_MartinLutherKingDay',
 'event_name_1_MemorialDay',
 "event_name_1_Mother's day",
 'event_name_1_NBAFinalsEnd',
 'event_name_1_NBAFinalsStart',
 'event_name_1_NewYear',
 'event_name_1_OrthodoxChristmas',
 'event_name_1_OrthodoxEaster',
 'event_name_1_Pesach End',
 'event_name_1_PresidentsDay',
 'event_name_1_Purim End',
 'event_name_1_Ramadan starts',
 'event_name_1_StPatricksDay',
 'event_name_1_SuperBowl',
 'event_name_1_Thanksgiving',
 'event_name_1_ValentinesDay',
 'event_na

### 8. Visual sanity check

Plot the target overlaid with two of the engineered features for one key. Lags should track the target with a delay, and rolling means should be smoother than the raw series.


In [12]:
import plotly.graph_objects as go
from utils.viz import _apply_theme, PALETTE

g = (train_features.groupby(KEY_COLS, observed=True)
                  .get_group(tuple(train_features[KEY_COLS].iloc[0]))
                  .sort_values(DATE_COL)
                  .tail(180))

fig = go.Figure()
fig.add_trace(go.Scatter(x=g[DATE_COL], y=g[TARGET_COL], name="actual",
                         line=dict(color=PALETTE["actual"], width=2)))
lag_col = f"{TARGET_COL}_lag_7"
roll_col = f"{TARGET_COL}_roll_mean_28"
if lag_col in g.columns:
    fig.add_trace(go.Scatter(x=g[DATE_COL], y=g[lag_col], name="lag 7",
                             line=dict(color=PALETTE["forecast"], width=1.5, dash="dot")))
if roll_col in g.columns:
    fig.add_trace(go.Scatter(x=g[DATE_COL], y=g[roll_col], name="rolling mean 28",
                             line=dict(color=PALETTE["forecast2"], width=1.5)))
_apply_theme(fig, title="Target with engineered features (one forecast key)", height=380,
             xaxis_title="Date", yaxis_title=TARGET_COL).show()


### 9. What about exogenous variables?

If your dataset has additional columns (price, promotion flag, weather), the `FeatureEngineer` keeps them untouched. Just merge them onto your panel **before** calling `transform`. The only requirement is that exogenous values are **known at prediction time** — that is the definition of a causal exogenous regressor.

### Recap

* All temporal features are computed **per forecast key**, with an explicit **lag of 1** before any rolling.
* Calendar features are deterministic and therefore always causal.
* Categorical keys are stored as `pandas.Categorical` — LightGBM handles them natively.
* `FeatureEngineer` is the single source of truth for the rest of the tutorial.
